# ETLegacy Core Infrastructure

Generate a small Einstein Toolkit-style thorn and inspect its CCL and source files.

Navigation: [Index](../index.ipynb) | Previous: [BHaH Project Anatomy](bhah_project_anatomy.ipynb) | Next: [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)

## Why This Matters
ETLegacy writes Einstein Toolkit-style thorns. A thorn is a component described by CCL files and source files.

## What You Will See
You will see complete CCL files, make.code.defn, and a complete generated source function for a small thorn.

Prerequisite bridge: this notebook follows [BHaH Project Anatomy](bhah_project_anatomy.ipynb). Terms are defined here before they are used.

## Generate a Small Thorn
CCL files declare a thorn interface, parameters, schedule, and source list. The source function here is a compact right-hand-side example.

In [1]:
from pathlib import Path
import contextlib
import io
import re
import tempfile

import nrpy.c_function as cfc
import nrpy.grid as gri
import nrpy.params as par
from nrpy.infrastructures.ETLegacy import (
    CodeParameters,
    MoL_registration,
    Symmetry_registration,
    boundary_conditions,
    interface_ccl,
    make_code_defn,
    param_ccl,
    schedule_ccl,
    simple_loop,
    zero_rhss,
)

workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_tutorial_etlegacy_", dir=Path.cwd())
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / "toy_etlegacy"
THORN_NAME = "ToyETLegacyNRPy"

for name in [
    f"{THORN_NAME}_rhs_eval",
    f"{THORN_NAME}_zero_rhss",
    f"{THORN_NAME}_MoL_registration",
    f"{THORN_NAME}_Symmetry_registration_oldCartGrid3D",
    f"{THORN_NAME}_specify_Driver_BoundaryConditions",
]:
    cfc.CFunction_dict.pop(name, None)
for name in ["psi", "energy", "psi_rhs", "energy_rhs"]:
    gri.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "ETLegacy")
par.register_CodeParameter("CCTK_INT", __name__, "fd_order", 4)
par.register_CodeParameter("CCTK_REAL", __name__, "wave_speed", 1.0)
gri.register_gridfunctions(["psi"], group="EVOL")
gri.register_gridfunctions(["energy"], group="AUX")

body = f"DECLARE_CCTK_ARGUMENTS_{THORN_NAME}_rhs_eval;\nDECLARE_CCTK_PARAMETERS;\n"
body += CodeParameters.read_CodeParameters(list_of_tuples__thorn_CodeParameter=[(THORN_NAME, "wave_speed")], declare_invdxxs=True)
body += simple_loop.simple_loop(
    loop_body=(
        f"{gri.ETLegacyGridFunction.access_gf('psi_rhs')} = "
        f"wave_speed * ({gri.ETLegacyGridFunction.access_gf('energy')} - "
        f"{gri.ETLegacyGridFunction.access_gf('psi')});"
    ),
    loop_region="interior",
)

cfc.register_CFunction(
    subdirectory=THORN_NAME,
    includes=["cctk.h", "cctk_Arguments.h", "cctk_Parameters.h"],
    desc="Evaluate a toy right-hand side.",
    cfunc_type="void",
    name=f"{THORN_NAME}_rhs_eval",
    params="CCTK_ARGUMENTS",
    body=body,
    ET_thorn_name=THORN_NAME,
    ET_schedule_bins_entries=[(
        "MoL_CalcRHS",
        """
schedule FUNC_NAME in MoL_CalcRHS
{
  LANG: C
  READS: evol_variables(everywhere)
  READS: aux_variables(everywhere)
  WRITES: evol_variables_rhs(interior)
} "Evaluate toy RHS"
""",
    )],
    ET_current_thorn_CodeParams_used=["wave_speed"],
)
zero_rhss.register_CFunction_zero_rhss(THORN_NAME)
MoL_registration.register_CFunction_MoL_registration(THORN_NAME)
Symmetry_registration.register_CFunction_Symmetry_registration_oldCartGrid3D(THORN_NAME)
boundary_conditions.register_CFunction_specify_Driver_BoundaryConditions(thorn_name=THORN_NAME)

print("registered C functions:")
for name in sorted(cfc.CFunction_dict):
    print(name)

generation_output = io.StringIO()
with contextlib.redirect_stdout(generation_output):
    interface_ccl.construct_interface_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        inherits="Boundary Grid MethodofLines",
        USES_INCLUDEs="USES INCLUDE: Symmetry.h\nUSES INCLUDE: Boundary.h\n",
        is_evol_thorn=True,
    )
    param_ccl.construct_param_ccl(project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME)
    schedule_ccl.construct_schedule_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        STORAGE="""
STORAGE: evol_variables[3]
STORAGE: evol_variables_rhs[1]
STORAGE: aux_variables[1]
""",
    )
    make_code_defn.output_CFunctions_and_construct_make_code_defn(project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME)

cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", generation_output.getvalue())
cleaned = cleaned.replace(str(WORKSPACE), "<workspace>").replace(str(PROJECT_DIR), "project/toy_etlegacy")
print("generation output:")
for line in cleaned.splitlines():
    if line.strip():
        print(line.rstrip())

required_files = ["interface.ccl", "param.ccl", "schedule.ccl", "src/make.code.defn", f"src/{THORN_NAME}_rhs_eval.c"]
for relative_path in required_files:
    assert (PROJECT_DIR / THORN_NAME / relative_path).is_file(), relative_path

print("complete thorn inventory:")
for path in sorted((PROJECT_DIR / THORN_NAME).rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR / THORN_NAME))

for relative_path in required_files:
    print(f"\n--- {relative_path} ---")
    print((PROJECT_DIR / THORN_NAME / relative_path).read_text(encoding="utf-8", errors="replace"))

registered C functions:
ToyETLegacyNRPy_MoL_registration
ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D
ToyETLegacyNRPy_rhs_eval
ToyETLegacyNRPy_specify_Driver_BoundaryConditions
ToyETLegacyNRPy_zero_rhss
generation output:
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/interface.ccl...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/param.ccl...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/schedule.ccl...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_MoL_registration.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_rhs_eval.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c...[written]
Checking <workspace>/project/toy_etlegacy/ToyETLegacyNRPy/sr

## Interpreting the Output
The printed CCL and source files show how the same registered C-function idea is expressed as Einstein Toolkit thorn metadata. The schedule entries state when each generated function runs.

## Where This Leads
- [C Function Registration](../1-intro/c_function.ipynb)
- [BHaH Project Anatomy](bhah_project_anatomy.ipynb)
- [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)
- [Backend Choice Guide](backend_choice_guide.ipynb)